# Practical Introduction to the Hugging Face Trainer
### Understanding the Architecture of the Training Orchestrator

The `transformers.Trainer` class provides a high-level API that abstracts away the complex PyTorch training loop. It handles:
- Device placement (single-GPU, multi-GPU DDP, Apple Silicon, CPU)
- Mixed precision (`fp16`/`bf16`) optimization
- Dynamic batch padding
- Periodic evaluation, logging, and checkpointing

This notebook demonstrates how to wire together each modular component—model, tokenizer, datasets, data collator, evaluation metrics, and hyperparameter arguments—without needing to run hours of model training.

## 0. Installation and Environment Setup
First, ensure required dependencies (`transformers`, `datasets`, `evaluate`, `accelerate`, and `scikit-learn`) are available.

In [1]:
import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

PyTorch version: 2.13.0
CUDA Available:  False


---
## 1. Load Pretrained Model & Tokenizer
We instantiate an encoder architecture (`distilbert-base-uncased`) with a sequence classification head configured for two classes (binary sentiment).

In [2]:
MODEL_NAME = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

print(f"Loaded model: {MODEL_NAME}")
print(f"Classification head initialized with {model.num_labels} target labels.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded model: distilbert/distilbert-base-uncased
Classification head initialized with 2 target labels.


---
## 2. Prepare & Tokenize Datasets
We load sample subsets of the `yelp_review_full` dataset for illustration and tokenize the text column.

> **Note on Truncation vs. Padding:** We only specify `truncation=True` during the mapping step and intentionally skip `padding=True`. Padding will be handled dynamically later.

In [3]:
# Load a small subset for demonstration purposes
dataset = load_dataset("yelp_review_full")
train_dataset = dataset["train"].select(range(1000))
eval_dataset = dataset["test"].select(range(200))

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

train_tokenized = train_dataset.map(preprocess_function, batched=True)
eval_tokenized = eval_dataset.map(preprocess_function, batched=True)

print(f"Tokenized train samples: {len(train_tokenized)}")
print(f"Tokenized eval samples:  {len(eval_tokenized)}")

Using the latest cached version of the dataset since yelp_review_full couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'yelp_review_full' at /Users/jagannathrao/.cache/huggingface/datasets/yelp_review_full/yelp_review_full/0.0.0/c1f9ee939b7d05667af864ee1cb066393154bf85 (last modified on Thu Sep  3 13:11:42 2026).


Tokenized train samples: 1000
Tokenized eval samples:  200


---
## 3. Dynamic Data Collator

### 💡 Key Setup Concept: Dynamic Padding vs. Static Padding
* **Static Padding (Inefficient)**: Padding every sequence in the dataset to the maximum sequence length (e.g., 512 tokens) beforehand wastes significant memory and GPU compute on zero-pad multiplications.
* **Dynamic Padding with `DataCollatorWithPadding` (Efficient)**: Pads each mini-batch strictly to the length of the longest sample *within that specific batch*. A batch containing only short sentences is padded to e.g. 64 tokens, saving substantial tensor operations.

In [4]:
# The collator uses the tokenizer's pad_token_id to dynamically pad mini-batches
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Initialized data collator for padding token ID: {tokenizer.pad_token_id}")

Initialized data collator for padding token ID: 0


---
## 4. Evaluation Metrics

### 💡 Key Setup Concept: `compute_metrics` Input Structure
* The `Trainer` collects model outputs during the evaluation loop into a named tuple `(predictions, label_ids)`.
* `predictions` are raw unnormalized model outputs (**logits**), not class labels or probabilities.
* Students must convert logits to discrete class predictions using `np.argmax(logits, axis=-1)` before comparing against the ground truth integer labels.

In [5]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert logits -> most probable class indices
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {**acc, **f1}

print("compute_metrics function registered successfully.")

compute_metrics function registered successfully.


---
## 5. Training Arguments Configuration

### 💡 Key Setup Concept: Modern Argument Naming (`eval_strategy`)
* Older versions of Transformers used `evaluation_strategy`.
* Modern versions use `eval_strategy` to maintain consistent naming with `save_strategy`.
* Using `load_best_model_at_end=True` along with `metric_for_best_model="f1"` ensures that once training completes, the trainer reloads the checkpoint that achieved the highest F1 score on the evaluation set.

In [6]:
use_fp16 = torch.cuda.is_available()  # fp16 requires CUDA GPU acceleration

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",            # Modern syntax (replaces 'evaluation_strategy')
    save_strategy="epoch",            # Match save cadence to evaluation
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,      # Automatically restore checkpoint with best validation metric
    metric_for_best_model="f1",       # Select best checkpoint by F1 score rather than loss
    greater_is_better=True,
    logging_strategy="epoch",
    logging_steps=50,
    fp16=use_fp16                     # Mixed precision flag
)

print("Training arguments successfully configured.")

Training arguments successfully configured.


---
## 6. Instantiating the Trainer

### 💡 Key Setup Concept: `processing_class` vs. `tokenizer`
* Historically, the argument passed to `Trainer` was `tokenizer=tokenizer`.
* In modern versions of `transformers`, this parameter has been generalized to `processing_class=tokenizer`. This supports unified handling of text tokenizers, audio feature extractors, and multi-modal image processors under the same API.

In [7]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    processing_class=tokenizer,       # Modern unified parameter (replaces tokenizer=tokenizer)
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer initialized and ready!")
print(f"Optimizer: {type(trainer.create_optimizer()).__name__}")

Trainer initialized and ready!
Optimizer: AdamW


---
## 7. Execution Workflow (Reference Only)
To initiate training and post-training evaluation, invoke the following methods:

```python
# 1. Start the complete training & validation loop
# trainer.train()

# 2. Run standalone evaluation on the test set
# eval_results = trainer.evaluate()
# print(eval_results)

# 3. Save model & tokenizer for production or Hugging Face Hub upload
# trainer.save_model("./my_finetuned_model")
# tokenizer.save_pretrained("./my_finetuned_model")
```